# 2.1 GROUP_COMPARE_MODELS - Leave-One-Out Cross-Validation

**Comprehensive group-level comparison of RL vs PROBE models:**
- In-Sample ΔBIC/ΔAIC (from individual fits)
- Out-of-Sample ΔBIC/ΔAIC (Leave-One-Out CV)
- Bootstrap 95% CI
- FDR-corrected significance testing
- Effect sizes (Cohen's d)
- Non-parametric tests (Wilcoxon)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, wilcoxon
import warnings
warnings.filterwarnings('ignore')

# CONFIG
MEG_ROOT = Path(r"E:\Develop\MEG")
OUT_DIR = MEG_ROOT / "GROUP_RESULTS_COMPARISON"
OUT_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_SUBJECTS = None
REQUIRED_BLOCKS = ("C", "S")
N_BOOTSTRAP = 10000
SEED = 42

# PER-SUBJECT FIT SUMMARIES

if INCLUDE_SUBJECTS is None:
    summary_files = sorted(MEG_ROOT.glob("*/behav/model_fit_summary.csv"))
else:
    summary_files = []
    for subj in INCLUDE_SUBJECTS:
        p = MEG_ROOT / subj / "behav" / "model_fit_summary.csv"
        if p.exists():
            summary_files.append(p)

if not summary_files:
    raise RuntimeError("No model_fit_summary.csv files found")

records = []
issues = []

for p in summary_files:
    subject = p.parent.parent.name
    df = pd.read_csv(p)

    need_cols = {"block", "model", "bic", "ll", "aic", "n"}
    if not need_cols.issubset(df.columns):
        issues.append(f"{subject}: missing columns {sorted(need_cols - set(df.columns))}")
        continue

    df["block"] = df["block"].astype(str)
    df["model"] = df["model"].astype(str).str.upper()
    df["bic"] = pd.to_numeric(df["bic"], errors="coerce")
    df["aic"] = pd.to_numeric(df["aic"], errors="coerce")
    df["ll"] = pd.to_numeric(df["ll"], errors="coerce")

    row = {"subject": subject}
    ok = True

    for block in REQUIRED_BLOCKS:
        d = df[df["block"] == block]
        if d.empty:
            issues.append(f"{subject}: no rows for block {block}")
            ok = False
            break

        try:
            probe_row = d.loc[d["model"] == "PROBE"].iloc[0]
            rl_row = d.loc[d["model"] == "RL"].iloc[0]

            bic_probe = float(probe_row["bic"])
            bic_rl = float(rl_row["bic"])
            aic_probe = float(probe_row["aic"])
            aic_rl = float(rl_row["aic"])
            ll_probe = float(probe_row["ll"])
            ll_rl = float(rl_row["ll"])
        except Exception as e:
            issues.append(f"{subject}: error in block {block}: {str(e)}")
            ok = False
            break

        row[f"bic_probe_{block}"] = bic_probe
        row[f"bic_rl_{block}"] = bic_rl
        row[f"aic_probe_{block}"] = aic_probe
        row[f"aic_rl_{block}"] = aic_rl
        row[f"ll_probe_{block}"] = ll_probe
        row[f"ll_rl_{block}"] = ll_rl
        row[f"delta_bic_{block}"] = bic_probe - bic_rl
        row[f"delta_aic_{block}"] = aic_probe - aic_rl

    if not ok:
        continue

    # Detect confirmatory (joint) fit:
    # In confirmatory mode the same joint BIC/AIC is written into
    # BOTH the C and S rows.  Summing them would double-count.
    # Detection: PROBE BIC identical across blocks AND fit_tau_mode
    # column (if present) says "confirmatory" / "beta_eps_separate".

    bic_probe_vals = [row[f"bic_probe_{b}"] for b in REQUIRED_BLOCKS if f"bic_probe_{b}" in row]
    aic_probe_vals = [row[f"aic_probe_{b}"] for b in REQUIRED_BLOCKS if f"aic_probe_{b}" in row]

    # Check fit_tau_mode column if it exists in the CSV
    tau_mode_vals = set()
    for b in REQUIRED_BLOCKS:
        d = df[(df["block"] == b) & (df["model"] == "PROBE")]
        if not d.empty and "fit_tau_mode" in d.columns:
            tm = str(d.iloc[0].get("fit_tau_mode", "")).strip().lower()
            if tm:
                tau_mode_vals.add(tm)

    joint_modes = {"confirmatory", "beta_eps_separate"}
    is_joint = bool(tau_mode_vals & joint_modes)

    # Also detect by BIC equality (fallback when column is absent)
    if not is_joint and len(bic_probe_vals) == 2:
        is_joint = abs(bic_probe_vals[0] - bic_probe_vals[1]) < 1e-6

    row["fit_mode"] = "confirmatory" if is_joint else "separate"

    if is_joint:
        # Joint fit: BIC/AIC already cover both blocks - use once
        row["bic_probe_total"] = row["bic_probe_C"]
        row["bic_rl_total"]    = sum(row[f"bic_rl_{b}"] for b in REQUIRED_BLOCKS)
        row["aic_probe_total"] = row["aic_probe_C"]
        row["aic_rl_total"]    = sum(row[f"aic_rl_{b}"] for b in REQUIRED_BLOCKS)
    else:
        # Separate fits: sum independently fitted BIC/AIC
        row["bic_probe_total"] = sum(row[f"bic_probe_{b}"] for b in REQUIRED_BLOCKS)
        row["bic_rl_total"]    = sum(row[f"bic_rl_{b}"] for b in REQUIRED_BLOCKS)
        row["aic_probe_total"] = sum(row[f"aic_probe_{b}"] for b in REQUIRED_BLOCKS)
        row["aic_rl_total"]    = sum(row[f"aic_rl_{b}"] for b in REQUIRED_BLOCKS)

    row["delta_bic_total"] = row["bic_probe_total"] - row["bic_rl_total"]
    row["delta_aic_total"] = row["aic_probe_total"] - row["aic_rl_total"]

    records.append(row)

subj_df = pd.DataFrame(records).sort_values("subject").reset_index(drop=True)
if subj_df.empty:
    raise RuntimeError("No valid subjects")

print(f"[✓] Loaded {len(subj_df)} subjects\n")

fit_modes = subj_df["fit_mode"].value_counts().to_dict() if "fit_mode" in subj_df.columns else {}
if "confirmatory" in fit_modes:
    print(f"[INFO] {fit_modes['confirmatory']} subject(s): JOINT fit (confirmatory) — BIC_total = BIC_joint (not summed)")
if "separate" in fit_modes:
    print(f"[INFO] {fit_modes['separate']} subject(s): SEPARATE fits — BIC_total = BIC_C + BIC_S")
print()

print(subj_df[["subject", "fit_mode", "delta_bic_C", "delta_bic_S", "delta_bic_total"]].to_string(index=False))

if issues:
    print("\n[WARN] Issues:")
    for x in issues[:5]:
        print(" -", x)
    if len(issues) > 5:
        print(f" ... and {len(issues)-5} more")

# STATISTICAL TESTS: In-Sample Analysis

def compute_effect_size(delta_values):
    """Cohen's d (effect vs mean=0)"""
    delta_values = np.asarray(delta_values, dtype=float)
    delta_values = delta_values[np.isfinite(delta_values)]
    if len(delta_values) < 2:
        return np.nan
    mean = np.mean(delta_values)
    std = np.std(delta_values, ddof=1)
    return mean / std if std > 0 else np.nan

def bootstrap_ci(values, n_bootstrap=10000, ci=0.95, seed=42):
    """Bootstrap 95% CI for mean"""
    np.random.seed(seed)
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)

    if n < 2:
        return np.nan, np.nan

    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(values, size=n, replace=True)
        bootstrap_means.append(np.mean(sample))

    alpha = 1 - ci
    lower = np.percentile(bootstrap_means, alpha/2 * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    return lower, upper

def analyze_comparison(delta_bic_values, delta_aic_values, metric_name):
    """Comprehensive statistical analysis"""
    delta_bic = np.asarray(delta_bic_values, dtype=float)
    delta_aic = np.asarray(delta_aic_values, dtype=float)
    delta_bic = delta_bic[np.isfinite(delta_bic)]
    delta_aic = delta_aic[np.isfinite(delta_aic)]

    n = len(delta_bic)

    results = {
        "metric": metric_name,
        "n": n,
        # BIC
        "mean_delta_bic": float(np.mean(delta_bic)),
        "std_delta_bic": float(np.std(delta_bic, ddof=1)) if n > 1 else np.nan,
        "cohens_d_bic": float(compute_effect_size(delta_bic)),
        "t_stat_bic": np.nan,
        "p_value_bic": np.nan,
        "p_value_wilcoxon_bic": np.nan,
        "ci_lower_bic": np.nan,
        "ci_upper_bic": np.nan,
        # AIC
        "mean_delta_aic": float(np.mean(delta_aic)),
        "std_delta_aic": float(np.std(delta_aic, ddof=1)) if n > 1 else np.nan,
        "cohens_d_aic": float(compute_effect_size(delta_aic)),
        "t_stat_aic": np.nan,
        "p_value_aic": np.nan,
        "p_value_wilcoxon_aic": np.nan,
        "ci_lower_aic": np.nan,
        "ci_upper_aic": np.nan,
    }

    if n >= 2:
        # BIC Analysis
        t_bic, p_bic = ttest_1samp(delta_bic, 0)
        results["t_stat_bic"] = float(t_bic)
        results["p_value_bic"] = float(p_bic)

        try:
            _, p_wilcoxon_bic = wilcoxon(delta_bic)
            results["p_value_wilcoxon_bic"] = float(p_wilcoxon_bic)
        except:
            pass

        ci_l, ci_u = bootstrap_ci(delta_bic, n_bootstrap=N_BOOTSTRAP)
        results["ci_lower_bic"] = float(ci_l)
        results["ci_upper_bic"] = float(ci_u)

        # AIC Analysis
        t_aic, p_aic = ttest_1samp(delta_aic, 0)
        results["t_stat_aic"] = float(t_aic)
        results["p_value_aic"] = float(p_aic)

        try:
            _, p_wilcoxon_aic = wilcoxon(delta_aic)
            results["p_value_wilcoxon_aic"] = float(p_wilcoxon_aic)
        except:
            pass

        ci_l, ci_u = bootstrap_ci(delta_aic, n_bootstrap=N_BOOTSTRAP)
        results["ci_lower_aic"] = float(ci_l)
        results["ci_upper_aic"] = float(ci_u)

    return results

# In-Sample Analysis
print("\n" + "="*80)
print("IN-SAMPLE ANALYSIS (from individual model fits)")
print("="*80)

analysis_results = []
for metric in ["delta_bic_total", "delta_bic_C", "delta_bic_S"]:
    bic_vals = subj_df[metric].to_numpy()
    aic_metric = metric.replace("bic", "aic")
    aic_vals = subj_df[aic_metric].to_numpy()

    result = analyze_comparison(bic_vals, aic_vals, metric)
    analysis_results.append(result)

analysis_df = pd.DataFrame(analysis_results)

# Display results
print("\nΔBIC (negative = PROBE better):")
print(analysis_df[["metric", "n", "mean_delta_bic", "std_delta_bic", "cohens_d_bic",
                    "t_stat_bic", "p_value_bic", "ci_lower_bic", "ci_upper_bic"]].to_string(index=False))

print("\nΔAIC (negative = PROBE better):")
print(analysis_df[["metric", "n", "mean_delta_aic", "std_delta_aic", "cohens_d_aic",
                    "t_stat_aic", "p_value_aic", "ci_lower_aic", "ci_upper_aic"]].to_string(index=False))

# FDR Correction
from scipy.stats import rankdata

def fdr_correction(pvalues, alpha=0.05):
    """Benjamini-Hochberg FDR"""
    pvalues = np.asarray(pvalues, dtype=float)
    mask = np.isfinite(pvalues)
    p_clean = pvalues[mask]

    n = len(p_clean)
    ranks = rankdata(p_clean)

    threshold_idx = np.where(p_clean <= alpha * ranks / n)[0]
    threshold = p_clean[threshold_idx[-1]] if len(threshold_idx) > 0 else 0

    return pvalues <= threshold, threshold

# Apply FDR correction to BIC p-values
bic_pvals = analysis_df["p_value_bic"].to_numpy()
fdr_reject_bic, fdr_threshold_bic = fdr_correction(bic_pvals, alpha=0.05)

# Apply FDR correction to AIC p-values
aic_pvals = analysis_df["p_value_aic"].to_numpy()
fdr_reject_aic, fdr_threshold_aic = fdr_correction(aic_pvals, alpha=0.05)

analysis_df["fdr_reject_bic"] = fdr_reject_bic
analysis_df["fdr_reject_aic"] = fdr_reject_aic

print(f"\n[FDR CORRECTION α=0.05]")
print(f"  BIC: threshold={fdr_threshold_bic:.2e}, {fdr_reject_bic.sum()}/3 significant")
print(f"  AIC: threshold={fdr_threshold_aic:.2e}, {fdr_reject_aic.sum()}/3 significant")

# SAVE OUTPUTS

subj_out = OUT_DIR / "in_sample_model_compare_subjects.csv"
analysis_out = OUT_DIR / "in_sample_model_compare_analysis.csv"
json_out = OUT_DIR / "in_sample_model_compare_stats.json"

subj_df.to_csv(subj_out, index=False, encoding="utf-8")
analysis_df.to_csv(analysis_out, index=False, encoding="utf-8")

json_out.write_text(
    json.dumps(analysis_results, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

print(f"\n[✓ SAVED]")
print(f"  {subj_out}")
print(f"  {analysis_out}")
print(f"  {json_out}")




In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

PARAM_COLS = [
    "lambda_actor_post", "lambda_alt1_post",
    "Q_actor_post", "gamma_actor_post",
    "lambda_entropy_norm_post", "lambda_top2_gap_post",
]

param_rows = []
for subj in (subj_df["subject"].tolist() if "subj_df" in dir() else []):
    for ses in ["C", "S"]:
        p = MEG_ROOT / subj / "behav" / "master_behavior_with_probe.csv"
        if not p.exists():
            continue
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        if "block" not in df.columns:
            continue
        d = df[df["block"].astype(str).str.upper() == ses]
        if d.empty:
            continue
        row = {"subject": subj, "session": ses, "n_trials": len(d)}
        for col in PARAM_COLS:
            if col in d.columns:
                x = pd.to_numeric(d[col], errors="coerce").dropna().to_numpy()
                row[col + "_mean"] = float(np.mean(x)) if x.size > 0 else np.nan
                row[col + "_std"]  = float(np.std(x, ddof=1)) if x.size > 1 else np.nan
        param_rows.append(row)

if param_rows:
    param_df = pd.DataFrame(param_rows)

    mean_cols = [c for c in param_df.columns if c.endswith("_mean")]
    std_cols  = {c.replace("_mean", ""): c.replace("_mean", "_std") for c in mean_cols}

    display_rows = []
    for _, r in param_df.iterrows():
        drow = {"Субъект": r["subject"], "Сессия": r["session"], "N триалов": int(r["n_trials"])}
        for mc in mean_cols:
            base = mc.replace("_mean", "").replace("_post", "").replace("lambda_", "λ_").replace("gamma_", "γ_").replace("Q_actor", "Q")
            mv = r.get(mc, np.nan)
            sv = r.get(std_cols.get(mc.replace("_mean",""), ""), np.nan)
            if np.isfinite(mv):
                drow[base] = f"{mv:.3f}" + (f" ±{sv:.3f}" if np.isfinite(sv) else "")
            else:
                drow[base] = "—"
        display_rows.append(drow)

    disp_df = pd.DataFrame(display_rows)

    print("=" * 80)
    print("mean ± std")
    print("=" * 80)
    display(
        disp_df.style
        .set_properties(**{"font-size": "12px", "text-align": "center"})
        .set_table_styles([
            {"selector": "th", "props": [("font-weight", "bold"), ("background-color", "#343a40"), ("color", "white"), ("text-align", "center")]},
            {"selector": "td", "props": [("border", "1px solid #dee2e6"), ("padding", "5px 8px")]},
            {"selector": "tr:hover td", "props": [("background-color", "#e8f4f8")]},
        ])
        .hide(axis="index")
    )

    # Групповые средние (по сессии)
    print()
    print("Group Averages (M ± SEM)")
    print("-" * 60)
    for ses in ["C", "S"]:
        d = param_df[param_df["session"] == ses]
        print(f"\n  Сессия {ses}  (N={len(d)} субъектов):")
        for mc in mean_cols:
            base = mc.replace("_mean", "").replace("_post", "")
            x = d[mc].dropna().to_numpy()
            if x.size == 0: continue
            sem = np.std(x, ddof=1) / np.sqrt(x.size) if x.size > 1 else np.nan
            print(f"    {base:35s}  {np.mean(x):.4f} ± {sem:.4f}")
else:
    print(" master_behavior_with_probe.csv wasn't found.")

In [ ]:

import numpy as np
import pandas as pd
from IPython.display import display

def _stars(p):
    if not np.isfinite(float(p)): return ""
    p = float(p)
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    if p < 0.10:  return "†"
    return "n.s."

sum_rows = []

for _, r in analysis_df.iterrows():
    metric = str(r["metric"])
    label_map = {
        "delta_bic_total": "ΔBIC PROBE−RL  (C+S total)",
        "delta_bic_C":     "ΔBIC PROBE−RL  (сессия C)",
        "delta_bic_S":     "ΔBIC PROBE−RL  (сессия S)",
    }
    label = label_map.get(metric, metric)
    n  = int(r["n"]) if np.isfinite(r["n"]) else "?"
    mb = r["mean_delta_bic"]; sb = r["std_delta_bic"]; db = r["cohens_d_bic"]
    tb = r["t_stat_bic"]; pb = r["p_value_bic"]; wb = r.get("p_value_wilcoxon_bic", np.nan)
    lb = r["ci_lower_bic"]; ub = r["ci_upper_bic"]
    sem_b = sb / np.sqrt(n) if np.isfinite(sb) and n > 0 else np.nan
    sum_rows.append({
        "Метрика": label,
        "N": n,
        "Mean ± SEM": f"{mb:.2f} ± {sem_b:.2f}" if np.isfinite(mb) and np.isfinite(sem_b) else "—",
        "Bootstrap 95%CI": f"[{lb:.2f}, {ub:.2f}]" if np.isfinite(lb) and np.isfinite(ub) else "—",
        "t (df=n−1)": f"{tb:.2f} ({n-1})" if np.isfinite(tb) else "—",
        "p (t-test)": f"{pb:.4f}" if np.isfinite(pb) else "—",
        "p (Wilcoxon)": f"{wb:.4f}" if np.isfinite(wb) else "—",
        "Cohen's d": f"{db:.2f}" if np.isfinite(db) else "—",
        "Sig": _stars(pb) if np.isfinite(pb) else "",
        "FDR": "✓" if bool(r.get("fdr_reject_bic", False)) else "✗",
    })

sum_df = pd.DataFrame(sum_rows)

def _highlight(row):
    sig = str(row.get("Sig", ""))
    if sig in ("***", "**", "*"): return ["background-color: #d4edda"] * len(row)
    if sig == "†":               return ["background-color: #fff3cd"] * len(row)
    return [""] * len(row)

print("=" * 80)
print("САММАРИ: сравнение моделей PROBE vs RL")
print(f"N субъектов: {len(subj_df)}")
print("Отрицательный ΔBIC = PROBE лучше RL (меньше = лучше)")
print("=" * 80)
print("★ *** p<.001  ** p<.01  * p<.05  † p<.10  n.s. р≥.10")
print()

display(
    sum_df.style
    .apply(_highlight, axis=1)
    .set_properties(**{"font-size": "13px", "text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("font-weight", "bold"), ("background-color", "#343a40"), ("color", "white")]},
        {"selector": "td", "props": [("border", "1px solid #dee2e6"), ("padding", "6px 10px")]},
    ])
    .hide(axis="index")
)

# PROBE win-rate
print()
print("─" * 60)
print("PROBE WIN-RATE по субъектам (ΔBIC < 0 → PROBE лучше):")
for metric, label in [("delta_bic_total","total"), ("delta_bic_C","C"), ("delta_bic_S","S")]:
    if metric not in subj_df.columns: continue
    x = subj_df[metric].dropna().to_numpy(float)
    wr = np.mean(x < 0)
    print(f"  {label:8s}: {wr:.0%}  ({int(np.sum(x<0))}/{len(x)} субъектов)")

print()
print("─" * 80)
print("ТЕКСТОВОЕ РЕЗЮМЕ (для отправки):")
print("─" * 80)
sig_rows  = sum_df[sum_df["Sig"].isin(["***","**","*"])]
ns_rows   = sum_df[sum_df["Sig"] == "n.s."]
trend_rows = sum_df[sum_df["Sig"] == "†"]

if not sig_rows.empty:
    print("\nЗНАЧИМЫЕ ЭФФЕКТЫ:")
    for _, r in sig_rows.iterrows():
        print(f"  ✓ {r['Метрика']}: {r['Mean ± SEM']}, {r['t (df=n−1)']}, p={r['p (t-test)']} {r['Sig']}, d={r[\"Cohen's d\"]}, FDR={r['FDR']}")
if not trend_rows.empty:
    print("\nТРЕНДЫ:")
    for _, r in trend_rows.iterrows():
        print(f"  ~ {r['Метрика']}: p={r['p (t-test)']} {r['Sig']}, d={r[\"Cohen's d\"]}")
if not ns_rows.empty:
    print("\nНЕ ЗНАЧИМО:")
    for _, r in ns_rows.iterrows():
        print(f"  ✗ {r['Метрика']}: p={r['p (t-test)']}, d={r[\"Cohen's d\"]}")
print()
print(f"CSV субъектов: {OUT_DIR / 'in_sample_model_compare_subjects.csv'}")
print(f"CSV анализа:   {OUT_DIR / 'in_sample_model_compare_analysis.csv'}")
